# Frequency-Wavenumber Spectra Diagnostic: LLC vs Emulators
Variables: KE = rho0/2 * (U^2 + V^2),  B = -g*sigma0/rho0

Outputs per variable:
1. Spectra grid: rows = depth k in [0,10,20,30,40,50], cols = LLC + emulators
2. Difference grid: rows = depth, cols = (LLC - emulator_n)
3. Error-vs-depth scatter: cols = emulators, x = mean/median diff, y = depth (0..50)

In [2]:
pip install fastjmd95

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 92.3 MB/s  0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.3
    Uninstalling numpy-2.4.3:
      Successfully uninstalled numpy-2.4.3
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ocean-emulators 1.0 requires numpy<2,>=1.26.4, but you have numpy 2.3.5 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import xrft
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from dask.diagnostics import ProgressBar
from fastjmd95 import jmd95numba

In [3]:
# # ============== LOAD LLC ==============
# llc_path = '/orcd/data/abodner/002/cody/LLC_patch/LLC4320_face1_i2880-3600_j720-1440.zarr'
# llc_patch_full = xr.open_zarr(llc_path)

llc_patch_full = xr.open_zarr('/orcd/data/abodner/003/LLC4320/LLC4320',consolidated=False).isel(
    face=1,
    i=slice(2880,3600),
    i_g=slice(2880,3600),
    j=slice(720,1440),
    j_g=slice(720,1440),
)
# quad: i=[2880:4320), j=[0:1440)
# Agulhas: i=[2880:3600), j=[720:1440)
# ============== LOAD EMULATORS ==============
emulator_configs = [
    {
        'name': 'rb-pred-resid-eager-ckpt-40',
        'key': 'emulator_1',
        'path': '/orcd/data/abodner/002/cody/inference_patch/2026-07-15-eval:Samudra_LLC:rb-Agulhas-pred_resid-reg-ckpt40-17993072/predictions_4d.zarr',
        'desc': ''
    },
    #     {
    #     'name': 'rb-pred-resid-reg-ckpt-25',
    #     'key': 'emulator_2',
    #     'path': '/orcd/data/abodner/002/cody/inference_patch/2026-07-13-eval:Samudra_LLC:rb-Agulhas-pred_resid-reg-ckpt25-17850330/predictions_4d.zarr',
    #     'desc': ''
    # },
    # {
    #     'name': 'rb-Agulhas-strides=1-pred_field-ckpt-25',
    #     'key': 'emulator_2',
    #     'path': '/orcd/data/abodner/002/cody/inference_patch/rb/2026-07-06-eval:Samudra_LLC:rb-Agulhas-strides=1-pred_field-ckpt-25-17335589/predictions_4d.zarr',
    #     'desc': ''   
    # },

]
emulator_patches_raw = {}
for cfg in emulator_configs:
    emulator_patches_raw[cfg['key']] = xr.open_dataset(cfg['path'], consolidated=True)
    print(f"Loaded {cfg['name']}: {cfg['desc']}")

def normalize_times(times):
    return pd.DatetimeIndex([
        pd.Timestamp(int(t.year), int(t.month), int(t.day),
                     int(t.hour), int(t.minute), int(t.second))
        if hasattr(t, 'year') else pd.Timestamp(t).floor('s')
        for t in times
    ])

llc_times_norm = normalize_times(llc_patch_full.time.values)
common_times = llc_times_norm
for cfg in emulator_configs:
    et = normalize_times(emulator_patches_raw[cfg['key']].time.values)
    common_times = common_times.intersection(et)
common_times = common_times.sort_values()

llc_mask = llc_times_norm.isin(common_times)
llc_patch = llc_patch_full.isel(time=llc_mask)
print(f"LLC subset to {len(common_times)} common times")

grid_vars = ['XC', 'YC', 'rA', 'Z']
emulator_patches = {}
for cfg in emulator_configs:
    pr = emulator_patches_raw[cfg['key']]
    pmask = normalize_times(pr.time.values).isin(common_times)
    p = pr.isel(time=pmask)
    for gv in grid_vars:
        p[gv] = llc_patch[gv]
    emulator_patches[cfg['key']] = p

emulator_info = [(cfg['name'], cfg['key']) for cfg in emulator_configs]
n_emulators = len(emulator_info)
all_patches = {'llc': llc_patch}
all_patches.update(emulator_patches)

print(f"\n=== Setup complete: LLC + {n_emulators} emulators ===")
for name, key in emulator_info:
    print(f"  {name} ({key})")

Loaded rb-pred-resid-eager-ckpt-40: 
LLC subset to 336 common times

=== Setup complete: LLC + 1 emulators ===
  rb-pred-resid-eager-ckpt-40 (emulator_1)


In [4]:
selected_time_range = [0, 240]
stepping = 1
start_idx, end_idx = selected_time_range

llc_patch = llc_patch.isel(time=slice(start_idx, end_idx + 1, stepping))

emulator_patches_subset = {}
for key, patch in emulator_patches.items():
    safe_end_idx = min(end_idx, patch.sizes['time'] - 1)
    emulator_patches_subset[key] = patch.isel(
        time=slice(start_idx, safe_end_idx + 1, stepping)
    )
emulator_patches = emulator_patches_subset

min_time_len = min([llc_patch.sizes['time']] +
                   [p.sizes['time'] for p in emulator_patches.values()])
llc_patch = llc_patch.isel(time=slice(0, min_time_len))
emulator_patches = {k: p.isel(time=slice(0, min_time_len))
                    for k, p in emulator_patches.items()}

all_patches = {'llc': llc_patch}
all_patches.update(emulator_patches)

print(f"Final synchronized length = {min_time_len}")
print(f"LLC: {llc_patch.sizes['time']} times")
for name, key in emulator_info:
    print(f"  {name} ({key}): {emulator_patches[key].sizes['time']} times")

Final synchronized length = 241
LLC: 241 times
  rb-pred-resid-eager-ckpt-40 (emulator_1): 241 times


In [5]:
def format_time(t_val):
    """Format a time value to DD/MM/YYYY:HH regardless of cftime or datetime64."""
    try:
        return f"{t_val.day:02d}/{t_val.month:02d}/{t_val.year}:{t_val.hour:02d}h"
    except AttributeError:
        t_pd = pd.Timestamp(t_val)
        return f"{t_pd.day:02d}/{t_pd.month:02d}/{t_pd.year}:{t_pd.hour:02d}h"

## Compute Kinetic Energy: KE = 1/2 * (U^2 + V^2)

In [6]:
for patch_name, patch in all_patches.items():
    print(f"Computing KE for {patch_name}...")
    U = patch['U'].values
    V = patch['V'].values
    KE = 0.5 * (U**2 + V**2)
    patch['KE'] = (('time', 'k', 'j', 'i'), KE)
    print(f"  ✓ KE shape: {KE.shape}")
print("Done.")

Computing KE for llc...
  ✓ KE shape: (241, 51, 720, 720)
Computing KE for emulator_1...
  ✓ KE shape: (241, 51, 720, 720)
Done.


## Compute Buoyancy: B = -g * sigma0 / rho0

In [ ]:
G = 9.81
rho0 = 1025

for patch_name, patch in all_patches.items():
    print(f"Computing Buoyancy for {patch_name}...")
    Salt = patch['Salt'].values
    Theta = patch['Theta'].values
    sigma0 = jmd95numba.rho(Salt, Theta, 0) - rho0
    B = -G * sigma0 / rho0
    patch['B'] = (('time', 'k', 'j', 'i'), B)
    print(f"  ✓ B shape: {B.shape}")
print("Done.")

Computing Buoyancy for llc...
  ✓ B shape: (16, 51, 720, 720)
Computing Buoyancy for emulator_1...
  ✓ B shape: (16, 51, 720, 720)
Computing Buoyancy for emulator_2...
  ✓ B shape: (16, 51, 720, 720)
Done.


# Let's make KE and buoyancy figures! Including: Surface fields, surface field errors, surface gradients, and errors w/ depth

In [6]:
os.makedirs('figs/KE_B/fields/', exist_ok=True)
# ============== SET VARIABLES HERE ==============
prog_vars = ['KE']#, 'B']
colormaps = {'KE': 'magma'}#, 'B': 'viridis'}
# ================================================
for var in prog_vars:
    print(f"Generating plots for {var}...")
    
    
    ref_patch = emulator_patches[emulator_info[0][1]]
    n_times = len(ref_patch.time)
    time_indices = list(range(n_times))
    nrows = len(time_indices)
    ncols_fields = 1 + n_emulators  # LLC + emulators
    ncols_diff = n_emulators         # emulators only
    
    cmap = colormaps[var]
    
    # ==================== PLOT 1: Surface fields ====================
    fig, axes = plt.subplots(nrows, ncols_fields, figsize=(3.6*ncols_fields, 3*nrows), dpi=200)
    
    if nrows == 1:
        axes = axes.reshape(1, -1)
    if ncols_fields == 1:
        axes = axes.reshape(-1, 1)
    
    for row, t in enumerate(time_indices):
        time_str = format_time(ref_patch.time.values[t])
        
        # Collect all surface fields
        fields = [llc_patch.isel(time=t, k=0)[var]]
        for emu_name, emu_key in emulator_info:
            fields.append(emulator_patches[emu_key].isel(time=t, k=0)[var])
        
        vmin = np.min([f.values.min() for f in fields])
        vmax = np.max([f.values.max() for f in fields])
        
        labels = ['LLC'] + [name for name, _ in emulator_info]
        
        for col, (field, label) in enumerate(zip(fields, labels)):
            ax = axes[row, col]
            cf = ax.contourf(field.coords.get('i', np.arange(field.shape[-1])),
                             field.coords.get('j', np.arange(field.shape[-2])), field,
                             cmap=cmap, vmin=vmin, vmax=vmax, levels=30)
            ax.set_title(f'{label} {var} {time_str}', fontsize=8)
            plt.colorbar(cf, ax=ax)
    
    plt.tight_layout()
    plt.savefig(f'figs/KE_B/fields/surface_{var}_fields.png')
    plt.close()
    
    # ==================== PLOT 2: Difference fields ====================
    fig, axes = plt.subplots(nrows, ncols_diff, figsize=(4*ncols_diff, 3*nrows), dpi=200)
    
    if nrows == 1 and ncols_diff > 1:
        axes = axes.reshape(1, -1)
    elif nrows > 1 and ncols_diff == 1:
        axes = axes.reshape(-1, 1)
    elif nrows == 1 and ncols_diff == 1:
        axes = axes.reshape(1, 1)
    
    for row, t in enumerate(time_indices):
        time_str = format_time(ref_patch.time.values[t])
        
        llc_vis = llc_patch.isel(time=t, k=0)[var]
        
        diffs = []
        for emu_name, emu_key in emulator_info:
            emu_vis = emulator_patches[emu_key].isel(time=t, k=0)[var]
            diffs.append(llc_vis.values - emu_vis.values)
        
        abs_max = np.max([np.abs(d).max() for d in diffs])
        vmin_d, vmax_d = -abs_max, abs_max
        
        row_axes = [axes[row, col] for col in range(ncols_diff)]
        
        for col, ((emu_name, _), diff) in enumerate(zip(emulator_info, diffs)):
            ax = row_axes[col]
            cf = ax.contourf(llc_vis.coords.get('i', np.arange(llc_vis.shape[-1])),
                             llc_vis.coords.get('j', np.arange(llc_vis.shape[-2])), diff,
                             cmap="bwr", vmin=vmin_d, vmax=vmax_d, levels=30)
            short_name = emu_name.replace('Emulator ', 'Em')
            ax.set_title(f'LLC - {short_name} {var} {time_str}', fontsize=8)
        
        fig.colorbar(cf, ax=row_axes, orientation='vertical',
                     fraction=0.046, pad=0.04)
    
    plt.savefig(f'figs/KE_B/fields/surface_{var}_differences.png')
    plt.close()
    
    print(f"✓ Saved plots for {var}")

Generating plots for KE...
✓ Saved plots for KE


In [7]:
grad_vars = ['KE']#, 'B']

for var in grad_vars:
    grad_name = f'grad_{var}'
    print(f"Computing {grad_name}...")
    
    for patch_name, patch in all_patches.items():
        data = patch[var].values  # (time, k, j, i)
        
        dx = np.sqrt(patch['rA'].values)  # (j, i) in meters
        dy = dx.copy()
        
        d_di = (np.roll(data, -1, axis=3) - np.roll(data, 1, axis=3)) / (2 * dx[np.newaxis, np.newaxis, :, :])
        d_dj = (np.roll(data, -1, axis=2) - np.roll(data, 1, axis=2)) / (2 * dy[np.newaxis, np.newaxis, :, :])
        
        grad_mag = np.sqrt(d_di**2 + d_dj**2)
        
        patch[grad_name] = (('time', 'k', 'j', 'i'), grad_mag)
        print(f"  ✓ {patch_name} {grad_name}: {grad_mag.shape}")

print("Done computing gradients!")

Computing grad_KE...
  ✓ llc grad_KE: (25, 51, 720, 720)
  ✓ emulator_1 grad_KE: (25, 51, 720, 720)
  ✓ emulator_2 grad_KE: (25, 51, 720, 720)
  ✓ emulator_3 grad_KE: (25, 51, 720, 720)
Done computing gradients!


In [8]:
gradient_masks = {}

for patch_name, patch in all_patches.items():
    gradient_masks[patch_name] = {}
    
    for var in grad_vars:
        grad_name = f'grad_{var}'
        grad_data = patch[grad_name].values  # (time, k, j, i)
        
        n_times, n_depths = grad_data.shape[0], grad_data.shape[1]
        mask = np.zeros_like(grad_data, dtype=bool)
        
        for t in range(n_times):
            for k in range(n_depths):
                field = grad_data[t, k]
                threshold = np.nanpercentile(field, 97.5)
                mask[t, k] = field >= threshold
        
        gradient_masks[patch_name][var] = mask
        print(f"✓ {patch_name} {var}: {mask.sum()} high-gradient pixels ({mask.sum() / mask.size * 100:.1f}%)")

print("Done creating gradient masks!")

✓ llc KE: 16522787 high-gradient pixels (2.5%)
✓ emulator_1 KE: 16524024 high-gradient pixels (2.5%)
✓ emulator_2 KE: 16524056 high-gradient pixels (2.5%)
✓ emulator_3 KE: 16524059 high-gradient pixels (2.5%)
Done creating gradient masks!


In [9]:
for var in grad_vars:
    print(f"Generating gradient drift figure for {var}...")
    
    os.makedirs(f'figs/KE_B/gradients/', exist_ok=True)
    
    ref_patch = emulator_patches[emulator_info[0][1]]
    n_times = len(ref_patch.time)
    time_indices = list(range(n_times))
    nrows = len(time_indices)
    ncols = n_emulators
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3.5*nrows), dpi=150)
    
    if nrows == 1 and ncols > 1:
        axes = axes.reshape(1, -1)
    elif nrows > 1 and ncols == 1:
        axes = axes.reshape(-1, 1)
    elif nrows == 1 and ncols == 1:
        axes = axes.reshape(1, 1)
    
    for row, t in enumerate(time_indices):
        time_str = format_time(ref_patch.time.values[t])
        
        llc_mask_surface = gradient_masks['llc'][var][t, 0]  # (j, i)
        
        for col, (emu_name, emu_key) in enumerate(emulator_info):
            ax = axes[row, col]
            
            emu_field = all_patches[emu_key].isel(time=t, k=0)[var].values
            emu_mask_surface = gradient_masks[emu_key][var][t, 0]
            
            ax.imshow(emu_field, cmap='Greys', aspect='auto', origin='lower')
            
            overlap_mask = llc_mask_surface & emu_mask_surface
            llc_only_mask = llc_mask_surface & ~emu_mask_surface
            emu_only_mask = emu_mask_surface & ~llc_mask_surface
            
            llc_j, llc_i = np.where(llc_only_mask)
            emu_j, emu_i = np.where(emu_only_mask)
            ovl_j, ovl_i = np.where(overlap_mask)
            
            ax.scatter(llc_i, llc_j, c='red', s=1, alpha=0.5, label='LLC top 2.5%', rasterized=True)
            ax.scatter(emu_i, emu_j, c='green', s=1, alpha=0.5, label='Emu top 2.5%', rasterized=True)
            ax.scatter(ovl_i, ovl_j, c='yellow', s=1, alpha=0.7, label='Overlap', rasterized=True)
            
            n_overlap = np.sum(overlap_mask)
            
            ax.set_title(f'{emu_name} {var} {time_str} overlap={n_overlap}', fontsize=8)
            ax.tick_params(labelsize=6)
            
            if row == 0 and col == 0:
                ax.legend(fontsize=5, loc='upper right', markerscale=5)
    
    plt.tight_layout()
    plt.savefig(f'figs/KE_B/gradients/surface_{var}_gradient_drift.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Saved gradient drift figure for {var}")

print("Done with gradient drift figures!")

Generating gradient drift figure for KE...
✓ Saved gradient drift figure for KE
Done with gradient drift figures!


In [10]:
depth_vars = ['KE']#, 'B']
ref_lines = {
    'KE': [0.5, 1.0],
  #  'B': [0.06, 0.12],
}

for var in depth_vars:
    print(f"Generating augmented depth error plots for {var}...")
    
    os.makedirs(f'figs/KE_B/depth_error/', exist_ok=True)
    
    n_depths = llc_patch.sizes['k']
    ref_patch = emulator_patches[emulator_info[0][1]]
    n_times = len(ref_patch.time)
    time_indices = list(range(n_times))
    
    nrows = len(time_indices)
    ncols = n_emulators
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3*nrows), dpi=150)
    
    if nrows == 1 and ncols > 1:
        axes = axes.reshape(1, -1)
    elif nrows > 1 and ncols == 1:
        axes = axes.reshape(-1, 1)
    elif nrows == 1 and ncols == 1:
        axes = axes.reshape(1, 1)
    
    depths = np.arange(n_depths)
    
    for row, t in enumerate(time_indices):
        time_str = format_time(ref_patch.time.values[t])
        
        llc_data = llc_patch.isel(time=t)[var].values  # (k, j, i)
        llc_grad_mask = gradient_masks['llc'][var][t]   # (k, j, i)
        
        # First pass: compute all errors for shared xlim
        row_mean_errors = []
        row_median_errors = []
        row_hg_mean_errors = []
        
        for emu_name, emu_key in emulator_info:
            emu_data = emulator_patches[emu_key].isel(time=t)[var].values
            diff = np.abs(llc_data - emu_data)
            
            diff_flat = diff.reshape(n_depths, -1)
            mean_errors = np.nanmean(diff_flat, axis=1)
            median_errors = np.nanmedian(diff_flat, axis=1)
            
            hg_mean_errors = np.zeros(n_depths)
            for k in range(n_depths):
                hg_pixels = diff[k][llc_grad_mask[k]]
                if len(hg_pixels) > 0:
                    hg_mean_errors[k] = np.nanmean(hg_pixels)
                else:
                    hg_mean_errors[k] = np.nan
            
            row_mean_errors.append(mean_errors)
            row_median_errors.append(median_errors)
            row_hg_mean_errors.append(hg_mean_errors)
        
        all_errors = np.concatenate(row_mean_errors + row_median_errors + row_hg_mean_errors)
        xmin = 0
        xmax = np.nanmax(all_errors) * 1.05
        
        # Second pass: plot
        for col, (emu_name, _) in enumerate(emulator_info):
            ax = axes[row, col]
            
            ax.scatter(row_mean_errors[col], depths, color='blue', s=30, alpha=0.7, zorder=3)
            ax.plot(row_mean_errors[col], depths, color='blue', alpha=0.4, linewidth=1.5, label='Mean')
            
            ax.scatter(row_median_errors[col], depths, color='red', s=30, alpha=0.7, zorder=3)
            ax.plot(row_median_errors[col], depths, color='red', alpha=0.4, linewidth=1.5, label='Median')
            
            ax.scatter(row_hg_mean_errors[col], depths, color='green', s=30, alpha=0.7, zorder=3)
            ax.plot(row_hg_mean_errors[col], depths, color='green', alpha=0.4, linewidth=1.5, label='HG Mean')
            
            for ref_val in ref_lines[var]:
                ax.axvline(x=ref_val, color='black', linestyle='--', linewidth=1.5, alpha=0.5, zorder=2)
            
            ax.set_title(f'{emu_name} {var} {time_str}', fontsize=8)
            ax.set_xlabel('Abs Error', fontsize=7)
            ax.set_ylabel('Depth (k)', fontsize=7)
            ax.set_ylim(n_depths - 1, 0)
            ax.set_xlim(xmin, xmax)
            ax.grid(alpha=0.2)
            ax.tick_params(labelsize=6)
            
            if col == 0:
                ax.legend(fontsize=6, loc='lower right')
    
    plt.tight_layout()
    plt.savefig(f'figs/KE_B/depth_error/{var}-depth_error_by_time.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Saved augmented depth error plots for {var}")

print("Done!")

Generating augmented depth error plots for KE...
✓ Saved augmented depth error plots for KE
Done!


## Spectra computation helpers

In [7]:
# Patch grid (LLC4320 face1 patch is ~720x720; we use uniform metric coords).
# We approximate dx from rA so that x1/y1 are in metres.

def build_metric_coords(patch):
    """Build approximate uniform x1/y1 coords in meters from rA mean."""
    dx = float(np.sqrt(np.nanmean(patch['rA'].values)))
    nj = patch.sizes['j']
    ni = patch.sizes['i']
    x1 = (np.arange(ni) - ni/2 + 0.5) * dx
    y1 = (np.arange(nj) - nj/2 + 0.5) * dx
    return x1, y1, dx


def azimuthal_avg(k, l, f, N, nfactor=4.0):
    """Azimuthally average 2-D spectrum slice f(l,k) -> 1-D in kr."""
    kk, ll = np.meshgrid(k, l)
    K = np.sqrt(kk**2 + ll**2)
    nbins = int(N / nfactor)
    if k.max() > l.max():
        ki = np.linspace(0., l.max(), nbins)
    else:
        ki = np.linspace(0., k.max(), nbins)
    kidx = np.digitize(K.ravel(), ki)
    area = np.bincount(kidx)
    kr = np.bincount(kidx, weights=K.ravel()) / np.maximum(area, 1)
    iso_f = np.ma.masked_invalid(
        np.bincount(kidx, weights=f.ravel()) / np.maximum(area, 1)
    ) * kr
    return kr, iso_f


def compute_iso_spectrum(da, x1, y1, time_s):
    """3-D power spectrum -> azimuthal average -> (omega, kr) in SI."""
    da = da.assign_coords(
        x1=('i', x1), y1=('j', y1), time=('time', time_s)
    ).swap_dims({'i': 'x1', 'j': 'y1'})

    # Drop leftover non-dim coords that share dims with x1/y1
    drop_list = [c for c in ['i', 'j', 'XC', 'YC', 'rA'] if c in da.coords]
    if drop_list:
        da = da.drop_vars(drop_list)

    da['x1'].attrs['units'] = 'm'
    da['y1'].attrs['units'] = 'm'
    da['time'].attrs['units'] = 's'
    da = da.chunk({'time': -1, 'y1': -1, 'x1': -1})

    with ProgressBar():
        ps3d = xrft.power_spectrum(
            da, dim=['x1', 'y1', 'time'],
            window=True, window_correction=True,
        ).compute()

    kx_vals = ps3d.freq_x1.values
    ky_vals = ps3d.freq_y1.values
    omega_vals = ps3d.freq_time.values
    nomega = len(omega_vals)
    nfactor = 4.0

    _kr, _ = azimuthal_avg(kx_vals, ky_vals,
                           ps3d.isel(freq_time=0).values,
                           len(kx_vals), nfactor)
    nkr = len(_kr)
    ps_iso = np.ma.zeros((nomega, nkr))
    for j in range(nomega):
        _, ps_iso[j, :] = azimuthal_avg(
            kx_vals, ky_vals,
            ps3d.isel(freq_time=j).values,
            len(kx_vals), nfactor,
        )
    _kr[0] = 0.0
    ps_iso_xr = xr.DataArray(
        np.array(ps_iso),
        coords={'freq_time': omega_vals, 'kr': _kr},
        dims=['freq_time', 'kr'],
    )
    ps_iso_xr['freq_time'].attrs['units'] = 'cycles/s'
    ps_iso_xr['kr'].attrs['units'] = 'cycles/m'
    return ps_iso_xr

## Compute spectra for KE and B at all depths for all patches

In [16]:
SPECTRA_VARS = ['KE']#, 'B']
K_LEVELS_FIG = [0, 10]#, 20, 30, 40, 50]   # for spectra grids
K_LEVELS_ALL = [0, 10]#, 20, 30, 40, 50]  # list(range(0, 51))         # for error vs depth scatter

x1, y1, dx_m = build_metric_coords(llc_patch)
print(f"Approx dx = {dx_m:.1f} m")

time_vals = llc_patch.time.values
time_s = (time_vals - time_vals[0]) / np.timedelta64(1, 's') \
    if np.issubdtype(time_vals.dtype, np.datetime64) else \
    np.array([(pd.Timestamp(t.isoformat()) - pd.Timestamp(time_vals[0].isoformat())).total_seconds()
              for t in time_vals])

spectra_cache = {}  # spectra_cache[var][patch_key][k] = ps_iso_xr

for var in SPECTRA_VARS:
    spectra_cache[var] = {}
    for patch_name, patch in all_patches.items():
        spectra_cache[var][patch_name] = {}
        for k in K_LEVELS_ALL:
            print(f"  spectra: {var} {patch_name} k={k}")
            da = patch[var].isel(k=k)
            spectra_cache[var][patch_name][k] = compute_iso_spectrum(da, x1, y1, time_s)

print("All spectra computed.")

Approx dx = 1621.4 m
  spectra: KE llc k=0
[###########                             ] | 27% Completed | 133.75 ms

/orcd/home/002/codycruz/LLC_ocean_emulator/high_res_diagnostics/.venv/lib64/python3.12/site-packages/xrft/xrft.py:47: FutureWarning: Please provide the name of window adhering to scipy.signal.windows. The boolean option will be deprecated in future releases.
  warnings.warn(
/orcd/home/002/codycruz/LLC_ocean_emulator/high_res_diagnostics/.venv/lib64/python3.12/site-packages/xrft/xrft.py:47: FutureWarning: Please provide the name of window adhering to scipy.signal.windows. The boolean option will be deprecated in future releases.
  warnings.warn(


[########################################] | 100% Completed | 27.63 ss
  spectra: KE llc k=10
[###########                             ] | 27% Completed | 137.67 ms

/orcd/home/002/codycruz/LLC_ocean_emulator/high_res_diagnostics/.venv/lib64/python3.12/site-packages/xrft/xrft.py:47: FutureWarning: Please provide the name of window adhering to scipy.signal.windows. The boolean option will be deprecated in future releases.
  warnings.warn(
/orcd/home/002/codycruz/LLC_ocean_emulator/high_res_diagnostics/.venv/lib64/python3.12/site-packages/xrft/xrft.py:47: FutureWarning: Please provide the name of window adhering to scipy.signal.windows. The boolean option will be deprecated in future releases.
  warnings.warn(


[########################################] | 100% Completed | 24.07 ss
  spectra: KE emulator_1 k=0
[                                        ] | 0% Completed | 17.84 ms

/orcd/home/002/codycruz/LLC_ocean_emulator/high_res_diagnostics/.venv/lib64/python3.12/site-packages/xrft/xrft.py:47: FutureWarning: Please provide the name of window adhering to scipy.signal.windows. The boolean option will be deprecated in future releases.
  warnings.warn(
/orcd/home/002/codycruz/LLC_ocean_emulator/high_res_diagnostics/.venv/lib64/python3.12/site-packages/xrft/xrft.py:47: FutureWarning: Please provide the name of window adhering to scipy.signal.windows. The boolean option will be deprecated in future releases.
  warnings.warn(


[########################################] | 100% Completed | 20.49 ss
  spectra: KE emulator_1 k=10
[                                        ] | 0% Completed | 11.44 ms

/orcd/home/002/codycruz/LLC_ocean_emulator/high_res_diagnostics/.venv/lib64/python3.12/site-packages/xrft/xrft.py:47: FutureWarning: Please provide the name of window adhering to scipy.signal.windows. The boolean option will be deprecated in future releases.
  warnings.warn(
/orcd/home/002/codycruz/LLC_ocean_emulator/high_res_diagnostics/.venv/lib64/python3.12/site-packages/xrft/xrft.py:47: FutureWarning: Please provide the name of window adhering to scipy.signal.windows. The boolean option will be deprecated in future releases.
  warnings.warn(


[########################################] | 100% Completed | 20.76 ss
All spectra computed.


In [18]:
K_LEVELS_FIG = [0, 10]#, 20, 30, 40, 50]

## Plot helpers

In [9]:
dt_hours = 1.0   # set once; or infer from data
nyquist_cph = 1.0 / (2.0 * dt_hours)

SPEC_XLIM = [0.01, 0.25] # 0.005, 0.25
SPEC_YLIM = [0.018, nyquist_cph] #0.018

VAR_LABELS = {
    'KE': r'$$KE\ [\mathrm{J/m^3}]$$',
    'B':  r'$$b\ [\mathrm{m/s^2}]$$',
}


def get_plot_arrays(ps):
    """Return kr_km(+), omega_cph(+), Z_vp = |w_cph|*P/1e3 in positive quadrant."""
    kr = ps.kr.values * 1e3
    om = ps.freq_time.values * 3600.0
    iso_km = ps.values / 1e3
    pos_o = om > 0
    pos_k = kr > 0
    Z = iso_km[np.ix_(pos_o, pos_k)]
    krp = kr[pos_k]
    omp = om[pos_o]
    Z_vp = np.abs(omp)[:, None] * Z
    return krp, omp, Z_vp

## Figure 1 (per variable): spectra grid — rows = depth, cols = LLC + emulators

In [10]:
os.makedirs('figs/KE_B/spectra', exist_ok=True)

panel_keys = ['llc'] + [key for _, key in emulator_info]
panel_labels = ['LLC'] + [name for name, _ in emulator_info]

for var in SPECTRA_VARS:
    print(f"Plotting spectra grid for {var}...")
    nrows = len(K_LEVELS_FIG)
    ncols = len(panel_keys)

    # Global vmin/vmax across all panels
    all_vals = []
    for k in K_LEVELS_FIG:
        for pkey in panel_keys:
            ps = spectra_cache[var][pkey][k]
            _, _, Z_vp = get_plot_arrays(ps)
            v = Z_vp[np.isfinite(Z_vp) & (Z_vp > 0)]
            if v.size > 0:
                all_vals.append(v)
    if len(all_vals) == 0:
        print('  no data; skipping')
        continue
    all_vals = np.concatenate(all_vals)
    vmin = np.percentile(all_vals, 1) #2
    vmax = np.percentile(all_vals, 99) #100
    levels = np.power(10., np.linspace(np.log10(vmin), np.log10(vmax), 16))[:15]

    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4.5*nrows), dpi=150)
    if nrows == 1: axes = axes.reshape(1, -1)
    if ncols == 1: axes = axes.reshape(-1, 1)

    for r, k in enumerate(K_LEVELS_FIG):
        for c, (pkey, plabel) in enumerate(zip(panel_keys, panel_labels)):
            ax = axes[r, c]
            ps = spectra_cache[var][pkey][k]
            krp, omp, Z_vp = get_plot_arrays(ps)
            cs = ax.contourf(
                krp, omp, Z_vp, levels=levels,
                norm=LogNorm(vmin=vmin, vmax=vmax),
                cmap='magma', extend='both',
            )
            ax.set_xscale('log'); ax.set_yscale('log')
            ax.set_xlim(SPEC_XLIM); ax.set_ylim(SPEC_YLIM)
            ax.set_title(f'{plabel}  k={k}', fontsize=12)
            if r == nrows - 1:
                ax.set_xlabel('kr (cycles/km)', fontsize=12)
            if c == 0:
                ax.set_ylabel('omega (cph)', fontsize=12)
            ax.tick_params(labelsize=7)

    cbar = fig.colorbar(cs, ax=axes.ravel().tolist(),
                        orientation='vertical', fraction=0.015, pad=0.02)
    cbar.set_label(rf'$|\omega|\,k_r\,P$ ({var})', fontsize=12)
    fig.suptitle(f'Frequency-Wavenumber Spectra — {var}', fontsize=12, y=0.995)
    out = f'figs/KE_B/spectra/spectra_grid_{var}.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'  ✓ saved {out}')

Plotting spectra grid for KE...


findfont: Font family ['STIXGeneral'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXGeneral'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXGeneral'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXGeneral'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXNonUnicode'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXNonUnicode'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXNonUnicode'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXSizeOneSym'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXSizeTwoSym'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXSizeThreeSym'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXSizeFourSym'] not found. Falling back to DejaVu Sans.
findfont: Font family ['STIXSizeFiveSym'] not found. Falling back to DejaVu Sans.
findfont: Font family ['cmsy10'] not

  ✓ saved figs/KE_B/spectra/spectra_grid_KE.png


In [11]:
from matplotlib.ticker import LogLocator, NullFormatter
from matplotlib.cm import ScalarMappable
import matplotlib.gridspec as gridspec


fig = plt.figure(figsize=(5.5*ncols, 3.5*nrows), dpi=150)
gs = gridspec.GridSpec(nrows, ncols, wspace=0.1, hspace=0.3)
axes = [fig.add_subplot(gs[0, c]) for c in range(ncols)]

norm = LogNorm(vmin=vmin, vmax=vmax)

for c, (pkey, plabel) in enumerate(zip(panel_keys, panel_labels)):
    ax = axes[c]
    ps = spectra_cache[var][pkey][0]
    krp, omp, Z_vp = get_plot_arrays(ps)

    cs = ax.contourf(
        krp, omp, Z_vp, levels=levels,
        norm=norm, cmap='magma', extend='both',
    )

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlim(SPEC_XLIM)
    ax.set_ylim(SPEC_YLIM)
    ax.set_xlabel('wavenumber (cpkm)', fontsize=14)

    # Clean log tick locators on both axes (decades only for major)
    ax.xaxis.set_major_locator(LogLocator(base=10.0, numticks=12))
    ax.xaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(2,10)*0.1, numticks=12))
    ax.xaxis.set_minor_formatter(NullFormatter())

    ax.yaxis.set_major_locator(LogLocator(base=10.0, numticks=12))
    ax.yaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(2,10)*0.1, numticks=12))
    ax.yaxis.set_minor_formatter(NullFormatter())

    if c == 0:
        ax.set_ylabel('frequency (cph)', fontsize=14)
    else:
        ax.set_ylabel('')
        # Hide BOTH major and minor y tick labels on right panels
        ax.tick_params(axis='y', which='both', labelleft=False)

    #ax.set_title(f'{plabel} (k=0, surface)', fontsize=12, fontweight='bold')
    ax.tick_params(labelsize=10)
    ax.grid(alpha=0.3, which='both', linestyle='--', linewidth=0.5)

# --- Colorbar via ScalarMappable so it's a proper continuous log bar ---
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
sm = ScalarMappable(norm=norm, cmap='magma')
sm.set_array([])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation='vertical', extend='neither')

# Pick decade ticks within [vmin, vmax]
lo = int(np.ceil(np.log10(vmin)))
hi = int(np.floor(np.log10(vmax)))
exps = np.arange(lo, hi + 1)

if len(exps) >= 2:
    cbar.set_ticks([10.0**e for e in exps])
    cbar.set_ticklabels([rf'$10^{{{e}}}$' for e in exps])
else:
    # Fallback: a few evenly spaced log ticks
    ticks = np.logspace(np.log10(vmin), np.log10(vmax), 5)
    cbar.set_ticks(ticks)
    cbar.set_ticklabels([f'{t:.1e}' for t in ticks])

cbar.ax.tick_params(labelsize=14)
cbar.set_label(rf'$|\omega|\,k_r\,P$ ({var})', fontsize=14, labelpad=5)

#fig.suptitle(f'Surface Frequency-Wavenumber Spectra — {var}',
#             fontsize=13, fontweight='bold', y=0.98)

out = f'figs/KE_B/spectra/spectra_surface_{var}.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'  ✓ saved {out}')

  ✓ saved figs/KE_B/spectra/spectra_surface_KE.png


## Figure 2 (per variable): difference grid — LLC - emulator_n, rows = depth

In [22]:
from matplotlib.colors import SymLogNorm
for var in SPECTRA_VARS:
    print(f'Plotting difference grid for {var}...')
    nrows = len(K_LEVELS_FIG)
    ncols = n_emulators

    # Compute all diffs to set symmetric vmax
    diff_vals = []
    diffs_cache = {}
    for k in K_LEVELS_FIG:
        diffs_cache[k] = {}
        krp, omp, Z_llc = get_plot_arrays(spectra_cache[var]['llc'][k])
        for _, emu_key in emulator_info:
            _, _, Z_emu = get_plot_arrays(spectra_cache[var][emu_key][k])
            d = Z_llc - Z_emu
            diffs_cache[k][emu_key] = (krp, omp, d)
            v = d[np.isfinite(d)]
            if v.size > 0:
                diff_vals.append(np.abs(v))
    if len(diff_vals) == 0:
        print('  no data; skipping')
        continue
    abs_max = np.percentile(np.concatenate(diff_vals), 98)
    linthresh = abs_max / 100  # adjust as desired

    norm = SymLogNorm(
        linthresh=linthresh,
        linscale=1.0,
        vmin=-abs_max,
        vmax=abs_max,
        base=10
    )

    neg = -np.logspace(np.log10(abs_max), np.log10(linthresh), 10)
    mid = np.linspace(-linthresh, linthresh, 5)
    pos = np.logspace(np.log10(linthresh), np.log10(abs_max), 10)

    levels = np.unique(np.concatenate([neg, mid, pos]))
    levels.sort()

    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4.5*nrows), dpi=150)
    if nrows == 1 and ncols > 1: axes = axes.reshape(1, -1)
    elif nrows > 1 and ncols == 1: axes = axes.reshape(-1, 1)
    elif nrows == 1 and ncols == 1: axes = axes.reshape(1, 1)

    for r, k in enumerate(K_LEVELS_FIG):
        for c, (emu_name, emu_key) in enumerate(emulator_info):
            ax = axes[r, c]
            krp, omp, d = diffs_cache[k][emu_key]
            cs = ax.contourf(
            krp,
            omp,
            d,
            levels=levels,
            cmap='bwr',
            norm=norm,
            extend='both')
            ax.set_xscale('log'); ax.set_yscale('log')
            ax.set_xlim(SPEC_XLIM); ax.set_ylim(SPEC_YLIM)
            ax.set_title(f'LLC - {emu_name}  k={k}', fontsize=9)
            if r == nrows - 1:
                ax.set_xlabel('kr (cycles/km)', fontsize=8)
            if c == 0:
                ax.set_ylabel('omega (cph)', fontsize=8)
            ax.tick_params(labelsize=7)

    cbar = fig.colorbar(cs, ax=axes.ravel().tolist(),
                        orientation='vertical', fraction=0.015, pad=0.02)
    cbar.set_label(f'Difference: LLC - emulator ({var})', fontsize=9)
    fig.suptitle(f'Frequency-Wavenumber Spectra Difference — {var}',
                 fontsize=12, y=0.995)
    out = f'figs/KE_B/spectra/spectra_diff_grid_{var}.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'  ✓ saved {out}')

Plotting difference grid for KE...
  ✓ saved figs/KE_B/spectra/spectra_diff_grid_KE.png


## Figure 3 (per variable): error vs depth scatter
Each emulator panel: x = mean / median |LLC_spectrum - emulator_spectrum|, y = depth k (0..50).

In [23]:
for var in SPECTRA_VARS:
    print(f'Plotting error-vs-depth for {var}...')
    fig, axes = plt.subplots(1, n_emulators,
                             figsize=(5*n_emulators, 7), dpi=150)
    if n_emulators == 1:
        axes = [axes]

    # First pass — collect for shared xlim
    err_data = {}  # emu_key -> (means, medians)
    for emu_name, emu_key in emulator_info:
        means = np.zeros(len(K_LEVELS_ALL))
        medians = np.zeros(len(K_LEVELS_ALL))
        for idx, k in enumerate(K_LEVELS_ALL):
            _, _, Z_llc = get_plot_arrays(spectra_cache[var]['llc'][k])
            _, _, Z_emu = get_plot_arrays(spectra_cache[var][emu_key][k])
            d = np.abs(Z_llc - Z_emu)
            d = d[np.isfinite(d)]
            means[idx] = np.nanmean(d) if d.size else np.nan
            medians[idx] = np.nanmedian(d) if d.size else np.nan
        err_data[emu_key] = (means, medians)

    all_err = np.concatenate([np.concatenate(v) for v in err_data.values()])
    all_err = all_err[np.isfinite(all_err)]
    xmin = 0
    xmax = float(np.nanmax(all_err)) * 1.05 if all_err.size else 1.0

    depths = np.array(K_LEVELS_ALL)
    for col, (emu_name, emu_key) in enumerate(emulator_info):
        ax = axes[col]
        means, medians = err_data[emu_key]
        ax.scatter(means, depths, color='blue', s=30, alpha=0.7,
                   label='Mean', zorder=3)
        ax.plot(means, depths, color='blue', alpha=0.4, linewidth=1.5)
        ax.scatter(medians, depths, color='red', s=30, alpha=0.7,
                   label='Median', zorder=3)
        ax.plot(medians, depths, color='red', alpha=0.4, linewidth=1.5)

        ax.set_title(f'{emu_name}  {var}', fontsize=9)
        ax.set_xlabel('|LLC - emulator| spectrum diff', fontsize=8)
        ax.set_ylabel('Depth k', fontsize=8)
        ax.set_ylim(max(K_LEVELS_ALL), 0)
        ax.set_xlim(xmin, xmax)
        ax.grid(alpha=0.3)
        ax.tick_params(labelsize=7)
        if col == 0:
            ax.legend(fontsize=7, loc='lower right')

    fig.suptitle(f'Spectral error vs depth — {var}', fontsize=11, y=1.0)
    plt.tight_layout()
    out = f'figs/KE_B/spectra/spectra_error_vs_depth_{var}.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'  ✓ saved {out}')

print('All spectra diagnostic figures saved to figs/KE_B/spectra/')

Plotting error-vs-depth for KE...
  ✓ saved figs/KE_B/spectra/spectra_error_vs_depth_KE.png
Plotting error-vs-depth for B...
  ✓ saved figs/KE_B/spectra/spectra_error_vs_depth_B.png
All spectra diagnostic figures saved to figs/KE_B/spectra/
